In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests
import io
import zipfile
import re

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

Log loaded. Rows: 35
PROJECT_ROOT: C:\Users\mjbou\governance-framework


## IMF iMaPP Pipeline

**Source:** IMF Integrated Macroprudential Policy Database
**Access:** Automated direct ZIP download with date auto-detection
**Download instructions:** See `docs/instructions_data_maintenance.md` — IMF_IMAPP section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Macroprudential toolkit breadth (count of distinct instruments ever activated) | Macroeconomic policy framework quality | Primary tier 1 (structural proxy) |
| Toolkit breadth by category (borrower / capital / liquidity / provisioning) | Macroeconomic policy framework quality | Supporting detail |

### Measurement note
iMaPP records macroprudential *change events* (+1 tighten, -1 loosen), not stock.
This pipeline measures CUMULATIVE ENGAGEMENT BREADTH: the number of distinct
instruments a country has ever taken action on, up to each year. This is a proxy
for how developed a country's macroprudential apparatus is — it is NOT a measure
of instruments currently in force, nor of regulatory quality directly.
Policy direction (tightening vs loosening) is deliberately excluded — the framework
cares about framework development, not policy stance or churn.
The instrument "Other" is excluded as too heterogeneous to interpret.
True framework quality assessment is deferred to IMF FSAP (Category 1 PDF work).

In [2]:
import requests
import zipfile
import io
import re
from datetime import datetime, timedelta
import pandas as pd

IMAPP_BASE = "https://www.elibrary-areaer.imf.org/Macroprudential/Documents"

def get_latest_imapp_url():
    """Auto-detect latest iMaPP ZIP by iterating dates backwards from today."""
    check_date = datetime.today()
    for _ in range(730):  # Search up to 2 years back
        date_str = check_date.strftime("%Y-%m-%d")
        url = f"{IMAPP_BASE}/iMaPP_database-{date_str}.zip"
        try:
            r = requests.head(url, timeout=5, allow_redirects=True)
            if r.status_code == 200 and 'zip' in r.headers.get('Content-Type', '').lower():
                print(f"Found latest iMaPP: {date_str}")
                return url, date_str
        except:
            pass
        check_date -= timedelta(days=1)
    return None, None

IMAPP_URL, IMAPP_DATE = get_latest_imapp_url()

if IMAPP_URL:
    print(f"\nDownloading iMaPP {IMAPP_DATE}...")
    response = requests.get(IMAPP_URL, timeout=120)
    print(f"Status: {response.status_code}, Size: {len(response.content)/1024/1024:.1f}MB")
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        print(f"ZIP contents: {z.namelist()}")

Found latest iMaPP: 2025-09-29

Status: 200, Size: 7.9MB
ZIP contents: ['Alam et al. (2019) iMaPP WP.pdf', 'iMaPP_database-2025-9-29.xlsx', 'iMaPP_load.do', 'ReadMe_Main.txt', 'Sample Files for Figures Alam et al. (2019)/', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig123.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig1234.xlsx', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig4.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5.do', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5a.emf', 'Sample Files for Figures Alam et al. (2019)/iMaPP_Fig5b.emf', 'Sample Files for Figures Alam et al. (2019)/iMaPP_LTV_average_statistics.docx', 'Sample Files for Figures Alam et al. (2019)/ReadMe_SampleFiles.txt', 'Sample Files for Figures Alam et al. (2019)/Thumbs.db']


In [10]:
# Build cumulative macroprudential toolkit breadth from the MaPP sheet
# MaPP codes each instrument-month: +1 tighten, -1 loosen, 0 no action.
# We treat ANY non-zero as "engaged with this instrument" (direction ignored).
import pandas as pd
import io, zipfile
from datetime import datetime

# Load the combined MaPP sheet (directionless action indicators) from the downloaded ZIP
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    excel_name = [f for f in z.namelist() if f.endswith('.xlsx') and 'iMaPP_database' in f][0]
    mapp = pd.read_excel(io.BytesIO(z.read(excel_name)), sheet_name='MaPP', engine='openpyxl')

# 16 instruments grouped into categories (instrument "Other"/OT excluded as too heterogeneous).
# RR flagged in docs as noisy (IMF warns it mixes monetary + macroprudential purposes).
instrument_categories = {
    'borrower_based':      ['LTV', 'DSTI', 'LoanR', 'LCG'],
    'capital_based':       ['CCB', 'Conservation', 'Capital', 'LVR', 'SIFI'],
    'liquidity_funding':   ['Liquidity', 'LTD', 'LFX', 'LFC'],
    'provision_reserve_tax': ['LLP', 'RR', 'Tax'],
}
all_instruments = [inst for group in instrument_categories.values() for inst in group]

# Confirm every expected instrument column exists — fail loudly if the source renames one
missing = [c for c in all_instruments if c not in mapp.columns]
if missing:
    raise ValueError(f"Expected instrument columns missing from MaPP sheet: {missing}")

# Mark any non-zero monthly action as engagement (1) for each instrument
engaged = mapp[['iso3', 'Country', 'Year']].copy()
engaged.columns = ['country_code', 'country_name', 'year']
for inst in all_instruments:
    engaged[inst] = (mapp[inst].fillna(0) != 0).astype(int)

# Collapse months to country-year: did the country act on each instrument at all that year
cy = engaged.groupby(['country_code', 'country_name', 'year'])[all_instruments].max().reset_index()
cy = cy.sort_values(['country_code', 'year'])

# Cumulative "ever engaged" per instrument up to each year (cummax along time within country)
cum = cy.copy()
cum[all_instruments] = cy.groupby('country_code')[all_instruments].cummax()

# Total breadth = count of distinct instruments ever activated up to each year
cum['imapp_breadth_total'] = cum[all_instruments].sum(axis=1)

# Category breadth = count within each instrument group
for cat, insts in instrument_categories.items():
    cum[f'imapp_breadth_{cat}'] = cum[insts].sum(axis=1)

# Keep identifiers + derived breadth measures only (drop the per-instrument flags)
breadth_cols = ['imapp_breadth_total'] + [f'imapp_breadth_{c}' for c in instrument_categories]
imapp = cum[['country_code', 'country_name', 'year'] + breadth_cols].copy()

# Filter to framework start year
imapp = imapp[imapp['year'] >= FRAMEWORK_START_YEAR].copy()
imapp = imapp.sort_values(['country_name', 'year']).reset_index(drop=True)

print(f"Shape: {imapp.shape}")
print(f"Years: {imapp['year'].min()} — {imapp['year'].max()}")
print(f"Countries: {imapp['country_name'].nunique()}")
print(f"\nTotal breadth distribution (latest year):")
latest = imapp[imapp['year'] == imapp['year'].max()]
print(latest['imapp_breadth_total'].value_counts().sort_index())
print(f"\nSample — a few high-breadth countries, latest year:")
print(latest.nlargest(5, 'imapp_breadth_total')[['country_name','year'] + breadth_cols].to_string())

Shape: (4725, 8)
Years: 1990 — 2024
Countries: 135

Total breadth distribution (latest year):
imapp_breadth_total
1      3
2      5
3      5
4      6
5      9
6      4
7     17
8     13
9     18
10    14
11    20
12    14
13     4
15     2
16     1
Name: count, dtype: int64

Sample — a few high-breadth countries, latest year:
     country_name  year  imapp_breadth_total  imapp_breadth_borrower_based  imapp_breadth_capital_based  imapp_breadth_liquidity_funding  imapp_breadth_provision_reserve_tax
944         China  2024                   16                             4                            5                                4                                    3
2309        Korea  2024                   15                             3                            5                                4                                    3
3324     Pakistan  2024                   15                             4                            5                                3              

In [11]:
# Derive data currency from ZIP filename date — no hardcoding
data_as_of = IMAPP_DATE[:7]  # YYYY-MM portion of the detected vintage date

# Save to processed
output_path = os.path.join(PROCESSED_DIR, "imapp_clean.csv")
imapp.to_csv(output_path, index=False)
print(f"Written: {output_path}")
print(f"Shape: {imapp.shape}")

# Update download log
update_entry(
    "IMF_IMAPP",
    last_successful_download_date=datetime.today().strftime("%Y-%m-%d"),
    data_as_of_date=data_as_of,
    local_filename="imapp_clean.csv",
    latest_available_version=IMAPP_DATE,
    notes="Cumulative macroprudential toolkit BREADTH: count of distinct instruments (of 16; 'Other' excluded) "
          "a country has ever activated, up to each year, total and by category (borrower-based, capital-based, "
          "liquidity-funding, provision-reserve-tax). Direction (tighten/loosen) excluded. "
          "Proxy for framework development, NOT instruments-in-force and NOT quality directly "
          "(quality assessment deferred to FSAP). RR is noisy per IMF (mixes monetary+macroprudential). "
          "Auto-detects latest ZIP by date iteration. Coverage: 1990-2024, 135 countries."
)
print_entry("IMF_IMAPP")

Written: C:\Users\mjbou\governance-framework\data\processed\imapp_clean.csv
Shape: (4725, 8)
[download_log] Updated entry for IMF_IMAPP
  source_id: IMF_IMAPP
  last_attempted_date: 2026-06-12
  last_successful_download_date: 2026-06-17
  data_as_of_date: 2025-09
  local_filename: imapp_clean.csv
  latest_available_version: 2025-09-29
  no_update_reason: nan
  notes: Cumulative macroprudential toolkit BREADTH: count of distinct instruments (of 16; 'Other' excluded) a country has ever activated, up to each year, total and by category (borrower-based, capital-based, liquidity-funding, provision-reserve-tax). Direction (tighten/loosen) excluded. Proxy for framework development, NOT instruments-in-force and NOT quality directly (quality assessment deferred to FSAP). RR is noisy per IMF (mixes monetary+macroprudential). Auto-detects latest ZIP by date iteration. Coverage: 1990-2024, 135 countries.
